In [1]:
#starting off and setting up
import pandas as pd
import numpy as np
from scipy.optimize import minimize

prices = pd.read_csv("prices.csv", index_col=0, parse_dates=True)
returns = prices.pct_change().dropna()
n_assets = len(returns.columns)

bounds = tuple((0, 1) for _ in range(n_assets))
initial = np.repeat(1/n_assets, n_assets)

In [2]:
#split the 10-year data into sample and testing ground
split_date = "2020-01-01"
sample = returns.loc[:split_date]
test=returns.loc[split_date:]

In [4]:
#get mean returns, covariances and set risk-free rate
sample_m = sample.mean() * 252
sample_cov = sample.cov() * 252
rfr = 0.02

#define function to get Sharpe ratio from given portfolio and maximise
def neg_sharpe_sample(w):
    r = np.sum(w * sample_m)
    v = np.sqrt(w.T @ sample_cov @ w)
    return -(r - rfr)/v

sum_to_one = {'type': 'eq', 'fun': lambda w: np.sum(w)-1}
sample_optimise = minimize(neg_sharpe_sample, initial, bounds=bounds, constraints=sum_to_one)
sample_w = sample_optimise.x

#get max Sharpe ratio and the portfolio producing it
print(-sample_optimise.fun)
print(pd.Series(sample_w, index=returns.columns).round(4))

1.497731739805984
AAPL    0.0325
CAT     0.0000
JNJ     0.0000
JPM     0.1612
KO      0.0849
MSFT    0.3068
NVDA    0.3196
PG      0.0775
WMT     0.0175
XOM     0.0000
dtype: float64


In [6]:
#now apply this optimised portfolio out-of-sample
oos_returns = test @ sample_w
oos_ann_return = oos_returns.mean() *252
oos_ann_vol = oos_returns.std() * np.sqrt(252)
oos_sharpe = (oos_ann_return-rfr)/oos_ann_vol
print(oos_ann_return, oos_ann_vol, oos_sharpe)

0.38506536974824274 0.29403873190011304 1.2415553807797604


In [7]:
#comapre against 1/n (naive)
equal_w = np.repeat(1/n_assets, n_assets)
ew_returns = test @ equal_w
ew_ann_return = ew_returns.mean() * 252
ew_ann_vol = ew_returns.std() * np.sqrt(252)
ew_sharpe = (ew_ann_return-rfr)/ew_ann_vol
print(ew_ann_return, ew_ann_vol, ew_sharpe)

0.24021938461574607 0.20280701183235694 1.0858568578377479
